In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=8):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Le MLP partagé, implémenté via des convolutions 1x1 pour conserver la dimension spatiale
        self.fc1   = nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        out = avg_out + max_out
        return self.sigmoid(out)

class TimeSubcarrierAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(TimeSubcarrierAttention, self).__init__()
        # Le papier utilise une convolution 7x7 pour fusionner le pool max et moyen
        padding = 3 
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Pooling le long de l'axe des canaux
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concaténation des deux représentations
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)

class CTSCAB(nn.Module):
    def __init__(self, in_channels):
        super(CTSCAB, self).__init__()
        # Séquence : Canal -> Temps/Sous-porteuse -> Canal
        self.ca1 = ChannelAttention(in_channels)
        self.tsa = TimeSubcarrierAttention()
        self.ca2 = ChannelAttention(in_channels)

    def forward(self, x):
        x = x * self.ca1(x)
        x = x * self.tsa(x)
        x = x * self.ca2(x)
        return x

In [3]:
class FeatureExtractor(nn.Module):
    def __init__(self, in_channels=1, num_filters=64):
        super(FeatureExtractor, self).__init__()
        
        layers = []
        current_channels = in_channels
        
        # Création des 5 blocs : Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d
        for _ in range(5):
            layers.append(nn.Conv2d(current_channels, num_filters, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(num_filters))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
            current_channels = num_filters
            
        self.cnn_blocks = nn.Sequential(*layers)
        
        # Intégration du bloc d'attention renforcé
        self.attention_block = CTSCAB(num_filters)

    def forward(self, x):
        # Extraction des features profondes
        x = self.cnn_blocks(x)
        
        # Raffinement via l'attention
        x = self.attention_block(x)
        
        # Aplatissement (Flatten) pour projeter dans l'espace d'embedding (pour le réseau prototypique)
        x = x.view(x.size(0), -1)
        return x

In [4]:
def euclidean_dist(x, y):
    """
    Calcule la distance euclidienne au carré entre les échantillons de requête et les prototypes.
    x : Tenseur de forme (N, D) - e.g., N requêtes
    y : Tenseur de forme (M, D) - e.g., M prototypes de classes
    """
    n = x.size(0)
    m = y.size(0)
    d = x.size(1)
    
    assert d == y.size(1)

    x = x.unsqueeze(1).expand(n, m, d)
    y = y.unsqueeze(0).expand(n, m, d)

    return torch.pow(x - y, 2).sum(2)

def prototypical_loss(query_features, support_features, support_labels, query_labels):
    """
    Calcule la perte du réseau prototypique (Eq 14 de l'Algorithme 1).
    """
    # Identifier les classes uniques dans ce batch
    classes = torch.unique(support_labels)
    n_classes = len(classes)
    
    # 1. Calcul des prototypes (centre de masse) pour chaque classe (C_k)
    prototypes = []
    for c in classes:
        # Sélectionner les features du support set appartenant à la classe c
        class_features = support_features[support_labels == c]
        # Calculer la moyenne (prototype)
        prototype = class_features.mean(dim=0)
        prototypes.append(prototype)
        
    prototypes = torch.stack(prototypes)
    
    # 2. Calcul des distances euclidiennes entre requêtes et prototypes
    dists = euclidean_dist(query_features, prototypes)
    
    # 3. Log-Softmax sur les distances (négatives car on veut minimiser la distance)
    log_p_y = F.log_softmax(-dists, dim=1)
    
    # 4. Trouver les indices correspondant aux vrais labels
    # (Création d'un dictionnaire pour mapper les vrais labels aux indices des prototypes)
    target_inds = torch.zeros_like(query_labels)
    for idx, c in enumerate(classes):
        target_inds[query_labels == c] = idx
        
    # 5. Calcul de la perte finale
    loss = F.nll_loss(log_p_y, target_inds)
    
    # Prédiction (classe avec la distance minimale)
    _, y_hat = log_p_y.max(1)
    acc = torch.eq(y_hat, target_inds).float().mean()
    
    return loss, acc

In [5]:
if __name__ == "__main__":
    # Paramètres simulés (ex: 5 échantillons pour un support set)
    batch_size = 5
    channels = 1 
    # CSI matrix shape: 270 subcarriers x 750 frames (comme détaillé dans l'expérience 2.1)
    subcarriers = 270
    frames = 750
    
    # Création d'un tenseur aléatoire simulant un batch de données CSI
    dummy_input = torch.randn(batch_size, channels, subcarriers, frames)
    
    # Initialisation du modèle
    model = FeatureExtractor(in_channels=channels)
    
    # Forward pass
    output_embeddings = model(dummy_input)
    
    print(f"Shape de l'entrée : {dummy_input.shape}")
    print(f"Shape des embeddings extraits : {output_embeddings.shape}")
    # Le modèle est prêt à être intégré dans la boucle d'entraînement (Algorithme 1 du papier)

Shape de l'entrée : torch.Size([5, 1, 270, 750])
Shape des embeddings extraits : torch.Size([5, 11776])
